# Lab 3 — N-Grams
## Ready-to-run VS Code / Jupyter version

This notebook covers:
- Unigram, Bigram, and Trigram
- Padding
- Maximum Likelihood Estimation (MLE)
- Bigram probability
- Perplexity
- **Task: Generating tweets using a Bigram MLE model**

**Important:** This version does **not** require the `emoji` package, `kagglehub`, or NLTK `punkt`.

## 1. Imports

The tokenizer used here is `wordpunct_tokenize`, so there is no need to download the NLTK `punkt` resource.

In [1]:
import re
import math
import random
import zipfile
import urllib.request
from pathlib import Path

import pandas as pd
import nltk

from nltk.tokenize import wordpunct_tokenize
from nltk.util import ngrams
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends

print("pandas:", pd.__version__)
print("nltk:", nltk.__version__)
print("Imports completed successfully.")

pandas: 2.2.3
nltk: 3.10.3
Imports completed successfully.


## 2. Unigram

A **unigram** contains one token.

\[
P(A)=\frac{Count(A)}{Count(all\ words)}
\]

In [2]:
sentence = "moses supposes his toeses are roses but moses supposes erroneously"
tokens = wordpunct_tokenize(sentence.lower())

unigram_list = list(ngrams(tokens, 1))
print("Unigrams:")
print(unigram_list)

unigram_counts = pd.Series(tokens).value_counts()
unigram_probabilities = unigram_counts / len(tokens)

print("\nUnigram probabilities:")
print(unigram_probabilities)

Unigrams:
[('moses',), ('supposes',), ('his',), ('toeses',), ('are',), ('roses',), ('but',), ('moses',), ('supposes',), ('erroneously',)]

Unigram probabilities:
moses          0.2
supposes       0.2
his            0.1
toeses         0.1
are            0.1
roses          0.1
but            0.1
erroneously    0.1
Name: count, dtype: float64


## 3. Bigram

A **bigram** contains two consecutive tokens.

\[
P(B\mid A)=\frac{Count(A,B)}{Count(A)}
\]

In [3]:
bigram_list = list(ngrams(tokens, 2))
print("Bigrams:")
print(bigram_list)

Bigrams:
[('moses', 'supposes'), ('supposes', 'his'), ('his', 'toeses'), ('toeses', 'are'), ('are', 'roses'), ('roses', 'but'), ('but', 'moses'), ('moses', 'supposes'), ('supposes', 'erroneously')]


## 4. Trigram

A **trigram** contains three consecutive tokens.

In [4]:
trigram_list = list(ngrams(tokens, 3))
print("Trigrams:")
print(trigram_list)

Trigrams:
[('moses', 'supposes', 'his'), ('supposes', 'his', 'toeses'), ('his', 'toeses', 'are'), ('toeses', 'are', 'roses'), ('are', 'roses', 'but'), ('roses', 'but', 'moses'), ('but', 'moses', 'supposes'), ('moses', 'supposes', 'erroneously')]


## 5. Padding

For a bigram model, one start token and one end token are added.

In [5]:
padded_bigrams = list(
    ngrams(
        pad_both_ends(tokens, n=2),
        n=2
    )
)

print("Padded bigrams:")
print(padded_bigrams)

Padded bigrams:
[('<s>', 'moses'), ('moses', 'supposes'), ('supposes', 'his'), ('his', 'toeses'), ('toeses', 'are'), ('are', 'roses'), ('roses', 'but'), ('but', 'moses'), ('moses', 'supposes'), ('supposes', 'erroneously'), ('erroneously', '</s>')]


## 6. Train a Bigram Maximum Likelihood Model

In [6]:
train_data, vocabulary = padded_everygram_pipeline(2, [tokens])

demo_lm = MLE(2)
demo_lm.fit(train_data, vocabulary)

print("P(supposes | moses) =", demo_lm.score("supposes", ["moses"]))
print("P(moses | supposes) =", demo_lm.score("moses", ["supposes"]))

P(supposes | moses) = 1.0
P(moses | supposes) = 0.0


## 7. Perplexity

Lower perplexity means the language model predicts the sample more confidently.

In [7]:
demo_test = [("moses", "supposes"), ("toeses", "are")]
print("Demo perplexity =", demo_lm.perplexity(demo_test))

Demo perplexity = 1.0


# Task — Generating Tweets Using N-Grams

Required work:

1. Load the dataset: **Large Random Tweets from Pakistan**  
   Kaggle source: `adizafar/large-random-tweets-from-pakistan`
2. Pre-process the tweets:
   - Remove hashtags
   - Remove `RT`
   - Remove websites / URLs
   - Remove mentions
   - Remove emojis
3. Build an **MLE Bigram** model.
4. Evaluate the model.
5. Calculate the probability of the bigram **(pakistan is)**.
6. Calculate the perplexity of the word **pakistan**.
7. Generate tweets using the trained Bigram model.

This notebook first tries to download the public Kaggle dataset directly with Python.
If Kaggle blocks the download, it uses a small clearly-labelled fallback corpus only so that the notebook can still run end-to-end.

## Task 1 — Load the data automatically

No `kagglehub` installation is needed.

In [8]:
DATA_DIR = Path("data_lab3")
DATA_DIR.mkdir(exist_ok=True)

KAGGLE_DOWNLOAD_URL = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "adizafar/large-random-tweets-from-pakistan"
)

zip_path = DATA_DIR / "pakistan_tweets.zip"

# Small fallback corpus. It is used ONLY if Kaggle cannot be downloaded.
# It is not presented as the Kaggle dataset.
fallback_tweets = [
    "Pakistan is a beautiful country with many cultures.",
    "Pakistan is full of talented young people.",
    "The weather in Pakistan is changing quickly today.",
    "Pakistan is preparing for an important match tonight.",
    "RT @news: Pakistan is discussing new plans https://example.com #Pakistan",
    "@user I think Pakistan is improving in many areas 😊",
    "Pakistan is known for its mountains and historic places.",
    "Lahore is a major city in Pakistan.",
    "Islamabad is the capital of Pakistan.",
    "Karachi is one of the largest cities in Pakistan.",
    "Pakistan is home to many languages and traditions.",
    "The northern areas of Pakistan are beautiful.",
    "Pakistan is working on new technology projects.",
    "Students in Pakistan are learning artificial intelligence.",
    "Pakistan is participating in international events.",
    "People are talking about Pakistan on social media.",
    "Pakistan is developing its digital economy.",
    "Education in Pakistan is an important topic.",
    "Pakistan is famous for cricket and passionate fans.",
    "Many tourists say Pakistan is worth visiting.",
    "Pakistan is rich in cultural heritage.",
    "The future of technology in Pakistan is promising.",
    "Pakistan is investing in young professionals.",
    "Pakistan is discussing education and innovation.",
    "Today Pakistan is in the news again.",
    "Many people believe Pakistan is changing.",
    "Pakistan is celebrating a national event today.",
    "The food of Pakistan is popular with visitors.",
    "Pakistan is connected to several neighboring countries.",
    "Pakistan is an important country in South Asia.",
    "The people of Pakistan are discussing current events.",
    "Pakistan is supporting new business ideas.",
    "Pakistan is focusing on digital skills.",
    "Pakistan is building new opportunities for students.",
    "Pakistan is seeing more interest in artificial intelligence.",
    "The technology sector in Pakistan is growing.",
    "Pakistan is improving access to online learning.",
    "Pakistan is encouraging innovation among students.",
    "Pakistan is working to improve public services.",
    "Pakistan is a country with diverse landscapes."
]

def find_csv_files(folder):
    return sorted(folder.rglob("*.csv"))

csv_files = find_csv_files(DATA_DIR)

if not csv_files:
    print("No local CSV found. Trying to download the Kaggle dataset...")
    try:
        req = urllib.request.Request(
            KAGGLE_DOWNLOAD_URL,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req, timeout=15) as response:
            zip_path.write_bytes(response.read())

        if not zipfile.is_zipfile(zip_path):
            raise RuntimeError("The downloaded file is not a valid ZIP archive.")

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(DATA_DIR)

        csv_files = find_csv_files(DATA_DIR)
        if not csv_files:
            raise RuntimeError("Dataset downloaded, but no CSV file was found inside it.")

        DATA_SOURCE = "Kaggle dataset"
        print("Kaggle dataset downloaded successfully.")

    except Exception as e:
        DATA_SOURCE = "Built-in fallback corpus"
        print("Kaggle download was unavailable on this computer.")
        print("Reason:", type(e).__name__, "-", str(e)[:180])
        print("Using the built-in fallback corpus so the lab can still run.")
else:
    DATA_SOURCE = "Local Kaggle CSV"
    print("Using existing CSV file(s) from:", DATA_DIR)

if csv_files:
    # Pick the largest CSV because the dataset may contain more than one file.
    csv_path = max(csv_files, key=lambda p: p.stat().st_size)
    print("Selected CSV:", csv_path)

    # Try common encodings.
    df = None
    for encoding in ["utf-8", "utf-8-sig", "latin1", "cp1252"]:
        try:
            df = pd.read_csv(csv_path, encoding=encoding, low_memory=False)
            break
        except UnicodeDecodeError:
            pass

    if df is None:
        raise RuntimeError("Could not read the CSV with common encodings.")
else:
    df = pd.DataFrame({"tweet": fallback_tweets})

print("\\nData source:", DATA_SOURCE)
print("Shape:", df.shape)
display(df.head())

Using existing CSV file(s) from: data_lab3
Selected CSV: data_lab3\Random Tweets from Pakistan- Cleaned- Anonymous.csv
\nData source: Local Kaggle CSV
Shape: (202202, 7)


,Unnamed: 0,created_at_tweet,full_text,retweet_count,favorite_count,reply_count,location
0,0,Wed Jul 03 07:20:15 +0000 2019,ØªÛØ±Ø§ ÙÛÚØ± Ù ÛØ±Ø§ ÙÛÚØ± ÙÙØ§Ø² Ø...,226,1097,127,"Punjab, Pakistan"
1,1,Fri Jul 02 02:50:53 +0000 2021,"Happy birthday to my brother n boss , May you ...",54,273,9,"Punjab, Pakistan"
2,2,Fri Jul 02 12:12:53 +0000 2021,â¤ï¸â¤ï¸,50,131,13,"Punjab, Pakistan"
3,3,Tue Oct 15 22:29:16 +0000 2019,`suspicious Â°jikook au jimin'in yaÅadÄ±ÄÄ± ...,522,2134,86,"Ä°stanbul, TÃ¼rkiye"
4,4,Wed May 13 04:34:31 +0000 2020,Speaking of @SpiderMan... ð https://t.co/...,74,535,11,Follow the ATP Tour â¡


## Task 2 — Detect the tweet column

Different versions of a dataset can use different column names, so this cell detects the most likely text column automatically.

In [9]:
print("Columns:")
print(list(df.columns))

preferred_columns = [
    "tweet", "tweets", "text", "content", "twitter_text",
    "twitter text", "message", "body"
]

lower_to_original = {str(c).strip().lower(): c for c in df.columns}

tweet_column = None
for candidate in preferred_columns:
    if candidate in lower_to_original:
        tweet_column = lower_to_original[candidate]
        break

if tweet_column is None:
    object_columns = [
        c for c in df.columns
        if df[c].dtype == "object" or pd.api.types.is_string_dtype(df[c])
    ]

    if not object_columns:
        raise ValueError("No text-like column was found in the dataset.")

    # Choose the text-like column with the largest average string length.
    average_lengths = {}
    for c in object_columns:
        sample = df[c].dropna().astype(str).head(5000)
        average_lengths[c] = sample.str.len().mean() if len(sample) else 0

    tweet_column = max(average_lengths, key=average_lengths.get)

print("Detected tweet column:", tweet_column)

tweets = df[tweet_column].dropna().astype(str)
tweets = tweets[tweets.str.strip().ne("")].reset_index(drop=True)

print("Number of non-empty tweets:", len(tweets))
print("\nExample:")
print(tweets.iloc[0])

Columns:
['Unnamed: 0', 'created_at_tweet', 'full_text', 'retweet_count', 'favorite_count', 'reply_count', 'location']
Detected tweet column: full_text
Number of non-empty tweets: 202151

Example:
ØªÛØ±Ø§ ÙÛÚØ± ÙÛØ±Ø§ ÙÛÚØ± ÙÙØ§Ø² Ø´Ø±ÛÙ ÙÙØ§Ø² Ø´Ø±ÛÙâ¤ï¸  Ú©ÛØ§ Ø¢Ù¾Ú©Ø§ Ø¨Ú¾Û Ú¾ÛØØØ @MaryamNSharif @Tanverhussan @SenPervaiz https://t.co/mDyOGs3sXv


## Task 3 — Pre-process the tweets

Required removals:
- hashtags
- RT
- URLs
- mentions
- emojis

In [10]:
# Emoji and symbol ranges handled with regex, so the external `emoji` package is NOT required.
EMOJI_PATTERN = re.compile(
    "["
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FAFF"
    "\u2600-\u26FF"
    "\u2700-\u27BF"
    "]+",
    flags=re.UNICODE
)

def clean_tweet(text):
    text = str(text)

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove RT as a standalone token
    text = re.sub(r"\bRT\b", " ", text, flags=re.IGNORECASE)

    # Remove mentions
    text = re.sub(r"@\w+", " ", text)

    # Remove hashtags completely (e.g. #Pakistan)
    text = re.sub(r"#\w+", " ", text)

    # Remove emojis / pictographic symbols
    text = EMOJI_PATTERN.sub(" ", text)

    # Normalize escaped slash used in some old Twitter URL data
    text = text.replace("\\/", "/")

    # Lowercase
    text = text.lower()

    # Keep letters, apostrophes, and spaces
    text = re.sub(r"[^a-zA-Z'\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

cleaned_df = pd.DataFrame({"original_tweet": tweets})
cleaned_df["cleaned_tweet"] = cleaned_df["original_tweet"].apply(clean_tweet)
cleaned_df = cleaned_df[
    cleaned_df["cleaned_tweet"].str.len() > 0
].reset_index(drop=True)

cleaned_tweets = cleaned_df["cleaned_tweet"]

print("Tweets after cleaning:", len(cleaned_tweets))
display(cleaned_df.head(10))

Tweets after cleaning: 143442


,original_tweet,cleaned_tweet
0,"Happy birthday to my brother n boss , May you ...",happy birthday to my brother n boss may you ha...
1,`suspicious Â°jikook au jimin'in yaÅadÄ±ÄÄ± ...,suspicious jikook au jimin'in ya ad kasabada a...
2,Speaking of @SpiderMan... ð https://t.co/...,speaking of
3,"Alexa, skip to 2021",alexa skip to
4,Jb tm moon lover ho gy to sooraj to tappy ga n...,jb tm moon lover ho gy to sooraj to tappy ga na
5,Alexa skip to 2022 now,alexa skip to now
6,"But this was one of my first shoots, about 6 y...",but this was one of my first shoots about year...
7,Trying phoneix in 140 ping ð¥ https://t.co/C...,trying phoneix in ping
8,Cant sleep ð¥µð¥± So why not . https://t.co/...,cant sleep so why not
9,@n1ffyyy Ayy gonna miss you too ð¥º,ayy gonna miss you too


## Task 4 — Tokenize and split into training / testing data

In [11]:
tokenized_tweets = [
    [token.lower() for token in wordpunct_tokenize(tweet) if token.strip()]
    for tweet in cleaned_tweets
]

# Remove sequences that are too short to form a useful bigram.
tokenized_tweets = [tokens for tokens in tokenized_tweets if len(tokens) >= 2]

if len(tokenized_tweets) < 5:
    raise ValueError("Not enough cleaned tweets are available to train/test the model.")

random.Random(42).shuffle(tokenized_tweets)

split_index = max(1, int(len(tokenized_tweets) * 0.90))
if split_index >= len(tokenized_tweets):
    split_index = len(tokenized_tweets) - 1

train_sentences = tokenized_tweets[:split_index]
test_sentences = tokenized_tweets[split_index:]

print("Training tweets:", len(train_sentences))
print("Testing tweets:", len(test_sentences))
print("\nFirst tokenized tweet:")
print(train_sentences[0])

Training tweets: 119008
Testing tweets: 13224

First tokenized tweet:
['bees', 'do']


## Task 5 — Build the Bigram MLE Model

In [12]:
train_ngrams, train_vocab = padded_everygram_pipeline(2, train_sentences)

lm = MLE(2)
lm.fit(train_ngrams, train_vocab)

print("Bigram MLE model trained successfully.")
print("Vocabulary size:", len(lm.vocab))

Bigram MLE model trained successfully.
Vocabulary size: 71293


## Task 6 — Evaluate the model using Perplexity

With an unsmoothed MLE model, held-out perplexity can be `inf` if the test data contains an unseen bigram.
This is mathematically expected because MLE gives unseen bigrams probability 0.

In [13]:
test_bigrams = []

for sentence_tokens in test_sentences:
    padded_tokens = list(pad_both_ends(sentence_tokens, n=2))
    test_bigrams.extend(list(ngrams(padded_tokens, 2)))

model_perplexity = lm.perplexity(test_bigrams)

print("Number of test bigrams:", len(test_bigrams))
print("Bigram MLE test perplexity:", model_perplexity)

if math.isinf(model_perplexity):
    print(
        "\nNote: infinity is valid for an unsmoothed MLE model when the "
        "test set contains at least one unseen bigram."
    )

Number of test bigrams: 233517
Bigram MLE test perplexity: inf

Note: infinity is valid for an unsmoothed MLE model when the test set contains at least one unseen bigram.


## Task 7 — Probability of the bigram (pakistan is)

We calculate:

\[
P(is \mid pakistan)
\]

In [14]:
pakistan_is_probability = lm.score("is", ["pakistan"])

print("P(is | pakistan) =", pakistan_is_probability)

pakistan_count = lm.counts["pakistan"]
pakistan_is_count = lm.counts[["pakistan"]]["is"]

print("Count(pakistan) =", pakistan_count)
print("Count(pakistan, is) =", pakistan_is_count)

if pakistan_count > 0:
    manual_probability = pakistan_is_count / pakistan_count
    print("Manual check =", manual_probability)

P(is | pakistan) = 0.0501066940646487
Count(pakistan) = 12653
Count(pakistan, is) = 634
Manual check = 0.0501066940646487


## Task 8 — Perplexity of the word (pakistan)

Because the task asks for a single word, it is evaluated as a unigram in the trained language model.

In [15]:
pakistan_word_perplexity = lm.perplexity([("pakistan",)])

print("Perplexity of the word 'pakistan' =", pakistan_word_perplexity)

if math.isinf(pakistan_word_perplexity):
    print("'pakistan' was not present in the training vocabulary.")

Perplexity of the word 'pakistan' = 174.6844226665613


## Task 9 — Generate Tweets

The model starts from the sentence-start token `<s>`.

In [16]:
def generate_tweet(model, num_words=20, seed=42):
    generated = model.generate(
        num_words=num_words,
        text_seed=["<s>"],
        random_seed=seed
    )

    if isinstance(generated, str):
        generated = [generated]

    clean_tokens = [
        token for token in generated
        if token not in {"<s>", "</s>", "<UNK>"}
    ]

    return " ".join(clean_tokens).strip()

print("Generated tweets:\n")

for seed in [1, 7, 21, 42, 99]:
    generated_tweet = generate_tweet(lm, num_words=20, seed=seed)
    print(f"{seed}: {generated_tweet}")

Generated tweets:

1: chalo rahn he emphasized that it must say ameen thats the safety imran khan are you understand
7: i am to ban on it ' s a largest and be immediately restore electricity due to visit the imperative
21: dear student me i appreciate the small intro outing after garmi ma ' t eta vakf adana valisi sn
42: pls constantly fall off the tourism hockey because of about everything right now and other than
99: is best communities by taliban are licking his moral to learn how ruda director speaking crowd in seoul this post


# Final Task Summary

The notebook has completed all required steps:

- Loaded the Pakistan tweets dataset automatically when Kaggle access was available.
- Removed hashtags, RT, URLs, mentions, and emojis.
- Tokenized the cleaned tweets.
- Built an **MLE Bigram** model.
- Evaluated the model using perplexity.
- Calculated **P(is | pakistan)**.
- Calculated the perplexity of **pakistan**.
- Generated new tweet-like sequences using the trained model.

**Before submission:** check the `Data source:` output in the loading cell.  
For the actual assignment, it should ideally say **Kaggle dataset** or **Local Kaggle CSV**.  
If it says **Built-in fallback corpus**, the code is working, but Kaggle was unavailable on that computer.